<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Colab_Script/gcs_to_drive_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 GCS → Google Drive Transfer

Transfer all rendered images from `gs://pyblender-render-farm/RenderImages/` to `MyDrive/PyBlender/Compare/`

**Auth Strategy:**
- **GCS**: Service account key (`pyblender-e37593034bc1.json`)
- **Drive**: Colab's native Google Drive mount (your Drive Gmail account)

**Drive folder structure:**
```
PyBlender/Compare/
├── boy_01_PC_v2/
│   ├── viridis_colormap/
│   │   ├── boy01.png
│   │   ├── boy01_noisy.png
│   │   └── ...
│   ├── inferno_colormap/
│   └── ... (168 colormap folders)
├── boy_02_pc_v2/
└── ...
```

## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2 — Install Dependencies

In [2]:
!pip install -q google-cloud-storage

## Cell 3 — Load GCS Service Account Key

Load the service account JSON key from Colab **Secrets**.
Ensure you have a secret named `GCS_SERVICE_ACCOUNT_KEY` with the content of `pyblender.json`.

In [3]:
from google.colab import userdata
import os

GCS_KEY_PATH = '/tmp/pyblender.json'

if os.path.exists(GCS_KEY_PATH):
    print(f'✓ Key already exists at {GCS_KEY_PATH}')
else:
    try:
        key_content = userdata.get('GCS_SERVICE_ACCOUNT_KEY')
        with open(GCS_KEY_PATH, 'w') as f:
            f.write(key_content)
        print(f'✓ Saved secret to {GCS_KEY_PATH}')
    except userdata.SecretNotFoundError:
        print("✗ Secret 'GCS_SERVICE_ACCOUNT_KEY' not found!")
        print("Please add it to the 'Secrets' tab (key icon) on the left sidebar.")

✓ Saved secret to /tmp/pyblender.json


## Cell 4 — Transfer ALL Files from GCS → Drive

This downloads every file from the bucket and writes it directly to your mounted Google Drive.

- **Resume-safe**: Skips files that already exist on Drive
- **Set `FORCE_OVERWRITE = True`** to re-download everything

In [ ]:
import os
import time
from google.cloud import storage
from collections import defaultdict

# ─── Configuration ─────────────────────────────────────────
GCS_KEY_PATH = '/tmp/pyblender.json'
BUCKET_NAME = 'pyblender-render-farm'
GCS_BASE_PATH = 'RenderImages'
DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender_Render_Farm/RenderImages'
FORCE_OVERWRITE = False
PROGRESS_INTERVAL = 25
# ───────────────────────────────────────────────────────────

def sizeof_fmt(num_bytes):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if abs(num_bytes) < 1024.0:
            return f'{num_bytes:.1f} {unit}'
        num_bytes /= 1024.0
    return f'{num_bytes:.1f} TB'


def transfer_gcs_to_drive():
    # ── 1. Connect to GCS ────────────────────────────────
    print('🔗 Connecting to GCS...')
    client = storage.Client.from_service_account_json(GCS_KEY_PATH)
    bucket = client.bucket(BUCKET_NAME)

    try:
        next(bucket.list_blobs(max_results=1, prefix=GCS_BASE_PATH + '/'))
        print(f'✓ Connected to bucket: {BUCKET_NAME}')
    except StopIteration:
        print(f'⚠ Bucket is empty or prefix has no files')
        return
    except Exception as e:
        print(f'✗ Failed to access bucket: {e}')
        return

    # ── 2. List ALL blobs ────────────────────────────────
    print(f'\n📋 Listing all files under gs://{BUCKET_NAME}/{GCS_BASE_PATH}/...')
    all_blobs = []
    total_size = 0

    for blob in bucket.list_blobs(prefix=GCS_BASE_PATH + '/'):
        if blob.name.endswith('/'):
            continue
        all_blobs.append(blob)
        total_size += blob.size or 0

    print(f'   Found {len(all_blobs)} files ({sizeof_fmt(total_size)})')

    if not all_blobs:
        print('Nothing to transfer!')
        return

    # ── 3. Analyze structure ─────────────────────────────
    pc_types = defaultdict(lambda: defaultdict(int))
    for blob in all_blobs:
        parts = blob.name[len(GCS_BASE_PATH) + 1:].split('/')
        if len(parts) >= 2:
            pc_types[parts[0]][parts[1]] += 1

    print(f'\n📂 Point Cloud Types found:')
    for pc_type, colormaps in sorted(pc_types.items()):
        total_files = sum(colormaps.values())
        print(f'   ├── {pc_type}: {len(colormaps)} colormaps, {total_files} files')

    # ── 4. Create Drive directory ────────────────────────
    os.makedirs(DRIVE_BASE_PATH, exist_ok=True)
    print(f'\n📁 Drive target: {DRIVE_BASE_PATH}')

    # ── 5. Transfer files ────────────────────────────────
    print(f'\n🚀 Starting transfer...\n')

    transferred = 0
    skipped = 0
    failed = 0
    bytes_transferred = 0
    start_time = time.time()
    failed_files = []

    for i, blob in enumerate(all_blobs):
        relative_path = blob.name[len(GCS_BASE_PATH) + 1:]
        drive_path = os.path.join(DRIVE_BASE_PATH, relative_path)

        # Skip if exists (unless overwrite)
        if not FORCE_OVERWRITE and os.path.exists(drive_path):
            skipped += 1
            continue

        os.makedirs(os.path.dirname(drive_path), exist_ok=True)

        try:
            blob.download_to_filename(drive_path)
            transferred += 1
            bytes_transferred += blob.size or 0
        except Exception as e:
            failed += 1
            failed_files.append((relative_path, str(e)))
            print(f'   ✗ FAILED: {relative_path} — {e}')
            continue

        # Progress
        done = transferred + skipped + failed
        if done % PROGRESS_INTERVAL == 0 or done == len(all_blobs):
            elapsed = time.time() - start_time
            rate = transferred / elapsed if elapsed > 0 else 0
            eta = (len(all_blobs) - done) / rate if rate > 0 else 0
            print(
                f'   [{done}/{len(all_blobs)}] '
                f'✓ {transferred} transferred, ⏭ {skipped} skipped, ✗ {failed} failed '
                f'| {sizeof_fmt(bytes_transferred)} | {rate:.1f} files/s | ETA: {eta:.0f}s'
            )

    # ── 6. Summary ───────────────────────────────────────
    elapsed = time.time() - start_time
    print(f'\n{"="*60}')
    print(f'✅ TRANSFER COMPLETE')
    print(f'{"="*60}')
    print(f'   Transferred : {transferred} files ({sizeof_fmt(bytes_transferred)})')
    print(f'   Skipped     : {skipped} files (already on Drive)')
    print(f'   Failed      : {failed} files')
    print(f'   Total time  : {elapsed:.1f}s ({elapsed/60:.1f} min)')
    if elapsed > 0 and transferred > 0:
        print(f'   Avg speed   : {transferred/elapsed:.1f} files/s')
    print(f'   Drive path  : {DRIVE_BASE_PATH}')
    print(f'{"="*60}')

    if failed_files:
        print(f'\n⚠ Failed files:')
        for path, error in failed_files:
            print(f'   • {path}: {error}')

    return transferred, skipped, failed


# Run the transfer
transfer_gcs_to_drive()

🔗 Connecting to GCS...
✓ Connected to bucket: pyblender-render-farm

📋 Listing all files under gs://pyblender-render-farm/RenderImages/...
   Found 1980 files (2.6 GB)

📂 Point Cloud Types found:
   ├── boy_01_PC_v2: 180 colormaps, 1980 files

📁 Drive target: /content/drive/MyDrive/PyBlender_Render_Farm/RenderImages

🚀 Starting transfer...

   [25/1980] ✓ 25 transferred, ⏭ 0 skipped, ✗ 0 failed | 35.0 MB | 1.2 files/s | ETA: 1616s
   [50/1980] ✓ 50 transferred, ⏭ 0 skipped, ✗ 0 failed | 69.4 MB | 1.3 files/s | ETA: 1543s
   [75/1980] ✓ 75 transferred, ⏭ 0 skipped, ✗ 0 failed | 103.6 MB | 1.3 files/s | ETA: 1512s
   [100/1980] ✓ 100 transferred, ⏭ 0 skipped, ✗ 0 failed | 137.6 MB | 1.3 files/s | ETA: 1436s
   [125/1980] ✓ 125 transferred, ⏭ 0 skipped, ✗ 0 failed | 172.3 MB | 1.3 files/s | ETA: 1414s
   [150/1980] ✓ 150 transferred, ⏭ 0 skipped, ✗ 0 failed | 208.2 MB | 1.3 files/s | ETA: 1359s
   [175/1980] ✓ 175 transferred, ⏭ 0 skipped, ✗ 0 failed | 242.6 MB | 1.4 files/s | ETA: 1314s


## Cell 5 — Verify Transfer

Check what ended up on Drive

In [ ]:
import os
from collections import defaultdict

DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender/Compare'

print('🔍 Verifying Drive contents...\n')

stats = defaultdict(lambda: defaultdict(int))
total_files = 0
total_size = 0

for root, dirs, files_list in os.walk(DRIVE_BASE_PATH):
    for f in files_list:
        fpath = os.path.join(root, f)
        rel = os.path.relpath(fpath, DRIVE_BASE_PATH)
        parts = rel.split(os.sep)
        if len(parts) >= 2:
            stats[parts[0]][parts[1]] += 1
        total_files += 1
        total_size += os.path.getsize(fpath)

print(f'📂 {DRIVE_BASE_PATH}')
print(f'   Total: {total_files} files ({total_size / (1024*1024):.1f} MB)\n')

for pc_type in sorted(stats):
    colormaps = stats[pc_type]
    files_count = sum(colormaps.values())
    print(f'   📁 {pc_type}/')
    print(f'      {len(colormaps)} colormap folders, {files_count} files')

print(f'\n✅ Verification complete!')

In [ ]:
# @title 2.4 Generate Comparison Visualizations
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# 1. Pull settings from your existing config
try:
    from config import CONFIG
    DRIVE_BASE_PATH = CONFIG["DRIVE_BASE_PATH"]
    PC_TYPE = CONFIG["PC_TYPE"]
    COLORMAPS = CONFIG["COLORMAPS"]
except ImportError:
    # Fallback if config isn't in memory
    DRIVE_BASE_PATH = "/content/drive/MyDrive/PyBlender_Render_Farm"
    PC_TYPE = "boy_01_PC_v2"
    COLORMAPS = ['Accent', 'Accent_r', 'Blues', 'Blues_r', 'BrBG', 'BrBG_r', 'BuGn', 'BuGn_r', 'BuPu', 'BuPu_r', 'CMRmap', 'CMRmap_r', 'Dark2', 'Dark2_r', 'GnBu', 'GnBu_r', 'Grays', 'Grays_r', 'Greens', 'Greens_r', 'Greys', 'Greys_r', 'OrRd', 'OrRd_r', 'Oranges', 'Oranges_r', 'PRGn', 'PRGn_r', 'Paired', 'Paired_r', 'Pastel1', 'Pastel1_r', 'Pastel2', 'Pastel2_r', 'PiYG', 'PiYG_r', 'PuBu', 'PuBuGn', 'PuBuGn_r', 'PuBu_r', 'PuOr', 'PuOr_r', 'PuRd', 'PuRd_r', 'Purples', 'Purples_r', 'RdBu', 'RdBu_r', 'RdGy', 'RdGy_r', 'RdPu', 'RdPu_r', 'RdYlBu', 'RdYlBu_r', 'RdYlGn', 'RdYlGn_r', 'Reds', 'Reds_r', 'Set1', 'Set1_r', 'Set2', 'Set2_r', 'Set3', 'Set3_r', 'Spectral', 'Spectral_r', 'Wistia', 'Wistia_r', 'YlGn', 'YlGnBu', 'YlGnBu_r', 'YlGn_r', 'YlOrBr', 'YlOrBr_r', 'YlOrRd', 'YlOrRd_r', 'afmhot', 'afmhot_r', 'autumn', 'autumn_r', 'berlin', 'berlin_r', 'binary', 'binary_r', 'bone', 'bone_r', 'brg', 'brg_r', 'bwr', 'bwr_r', 'cividis', 'cividis_r', 'cool', 'cool_r', 'coolwarm', 'coolwarm_r', 'copper', 'copper_r', 'cubehelix', 'cubehelix_r', 'flag', 'flag_r', 'gist_earth', 'gist_earth_r', 'gist_gray', 'gist_gray_r', 'gist_grey', 'gist_grey_r', 'gist_heat', 'gist_heat_r', 'gist_ncar', 'gist_ncar_r', 'gist_rainbow', 'gist_rainbow_r', 'gist_stern', 'gist_stern_r', 'gist_yarg', 'gist_yarg_r', 'gist_yerg', 'gist_yerg_r', 'gnuplot', 'gnuplot2', 'gnuplot2_r', 'gnuplot_r', 'gray', 'gray_r', 'grey', 'grey_r', 'hot', 'hot_r', 'hsv', 'hsv_r', 'inferno', 'inferno_r', 'jet', 'jet_r', 'magma', 'magma_r', 'managua', 'managua_r', 'nipy_spectral', 'nipy_spectral_r', 'ocean', 'ocean_r', 'pink', 'pink_r', 'plasma', 'plasma_r', 'prism', 'prism_r', 'rainbow', 'rainbow_r', 'seismic', 'seismic_r', 'spring', 'spring_r', 'summer', 'summer_r', 'tab10', 'tab10_r', 'tab20', 'tab20_r', 'tab20b', 'tab20b_r', 'tab20c', 'tab20c_r', 'terrain', 'terrain_r', 'turbo', 'turbo_r', 'twilight', 'twilight_r', 'twilight_shifted', 'twilight_shifted_r', 'vanimo', 'vanimo_r', 'viridis', 'viridis_r', 'winter', 'winter_r']

# 2. Create the target Output Directory
compare_dir = os.path.join(DRIVE_BASE_PATH, "Compare", PC_TYPE)
os.makedirs(compare_dir, exist_ok=True)

# 3. File-to-Label Mapping & Order
# Edit this dictionary to match keywords in your filenames to the exact labels you want.
# The order of this dictionary dictates the left-to-right order of the images.
FILE_MAPPING = {
    "noisy": "Noisy",
    "bf": "BF",
    "wlop": "WLOP",
    "ad_": "AD",
    "dmr": "DMR",
    "score": "Score",
    "iterpfn": "IterPFN",
    "straightpcf": "StraightPCF",
    "delnoise": "De(l)Noise",
    "ours": "Ours",
    "gt": "GT"
}

def get_label_and_sort_key(filename):
    lower_name = filename.lower()
    for i, (key, label) in enumerate(FILE_MAPPING.items()):
        if key in lower_name:
            return label, i
    # Default fallback if keyword isn't found
    return filename.split('.')[0].title(), 99

# 4. Generate Visualizations for each Colormap
for colormap in COLORMAPS:
    render_dir = os.path.join(DRIVE_BASE_PATH, "RenderImages", PC_TYPE, f"{colormap}_colormap")

    if not os.path.exists(render_dir):
        print(f"Directory not found: {render_dir}. Skipping {colormap}...")
        continue

    # Grab all actual render images (ignoring the standalone gradient image and blend files)
    image_files = [f for f in os.listdir(render_dir) if f.endswith('.png') and 'colormap' not in f]

    if not image_files:
        print(f"No images found for {colormap}. Skipping...")
        continue

    # Sort files based on the order defined in FILE_MAPPING
    image_files.sort(key=lambda x: get_label_and_sort_key(x)[1])

    n_images = len(image_files)

    # Setup Figure: Width ratios ensure images get equal space, and colorbar gets a tiny sliver
    fig, axes = plt.subplots(1, n_images + 1, figsize=(n_images * 2.5, 4), gridspec_kw={'width_ratios': [1]*n_images + [0.15]})

    # Plot each render
    for i, img_name in enumerate(image_files):
        img_path = os.path.join(render_dir, img_name)
        img = Image.open(img_path)

        # Crop whitespace if Blender rendered with a large empty background (Optional)
        # img = img.crop(img.getbbox())

        axes[i].imshow(img)
        axes[i].axis('off')

        # Apply the clean label
        label, _ = get_label_and_sort_key(img_name)
        axes[i].text(0.5, -0.1, label, size=14, ha="center", va="top", transform=axes[i].transAxes, fontfamily='serif')

    # Draw the Vertical Colorbar on the last axis
    # Linspace from 1 to 0 so the highest values (brightest colors) are at the top
    gradient = np.linspace(1, 0, 256).reshape(256, 1)
    axes[-1].imshow(gradient, aspect='auto', cmap=colormap)
    axes[-1].axis('off')

    # Save and Output
    plt.tight_layout()
    out_path = os.path.join(compare_dir, f"{colormap}_comparison.png")

    # Facecolor='white' prevents transparent backgrounds from turning black in dark mode viewers
    plt.savefig(out_path, bbox_inches='tight', dpi=300, facecolor='white')
    print(f"Saved comparison for {colormap} at: {out_path}")

    # Display it inside the notebook
    # plt.show()

## Cell 6 (Optional) — Cross-Check: Find Missing Files

Compares bucket contents against Drive to ensure **nothing was left behind**

In [ ]:
import os
from google.cloud import storage

GCS_KEY_PATH = '/tmp/pyblender-e37593034bc1.json'
BUCKET_NAME = 'pyblender-render-farm'
GCS_BASE_PATH = 'RenderImages'
DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender/Compare'

client = storage.Client.from_service_account_json(GCS_KEY_PATH)
bucket = client.bucket(BUCKET_NAME)

missing = []
matched = 0

for blob in bucket.list_blobs(prefix=GCS_BASE_PATH + '/'):
    if blob.name.endswith('/'):
        continue
    relative_path = blob.name[len(GCS_BASE_PATH) + 1:]
    drive_path = os.path.join(DRIVE_BASE_PATH, relative_path)
    if os.path.exists(drive_path):
        matched += 1
    else:
        missing.append(relative_path)

print(f'✓ Matched on Drive: {matched}')
print(f'✗ Missing from Drive: {len(missing)}')

if missing:
    print(f'\nMissing files:')
    for m in missing[:50]:
        print(f'   • {m}')
    if len(missing) > 50:
        print(f'   ... and {len(missing) - 50} more')
else:
    print(f'\n✅ ALL bucket files are present on Drive! Nothing left behind.')